# CogAgent Vision Server for Pecifics AI
**Run this notebook on Kaggle with GPU T4 x2 enabled.**

This serves CogAgent-9B as a vision API that your desktop Electron app connects to via ngrok.

### Setup Steps:
1. Enable **GPU T4 x2** in Kaggle Settings (right panel)
2. Add your **ngrok auth token** as a Kaggle Secret named `NGROK_TOKEN`  
   Get free token: https://dashboard.ngrok.com/get-started/your-authtoken
3. Run all cells in order
4. Copy the ngrok URL and paste it into the Pecifics app settings

In [ ]:
# ============================================================
# Cell 1: Install Dependencies
# ============================================================
# NOTE: Do NOT reinstall torch/torchvision — Kaggle already has
# the correct CUDA-enabled PyTorch pre-installed.

import subprocess, sys

def safe_install(packages: str, extra_args: str = ""):
    """Install packages, but don't fail if network is down."""
    try:
        cmd = [sys.executable, "-m", "pip", "install", "-q"] + extra_args.split() + packages.split()
        cmd = [c for c in cmd if c]
        subprocess.check_call(cmd, timeout=300)
    except Exception as e:
        print(f"⚠️  pip install skipped ({type(e).__name__}). Using pre-installed packages.")

# Install transformers that matches Kaggle's pre-installed tokenizers.
# Kaggle ships tokenizers>=0.22, which needs transformers>=4.49.
# --no-deps prevents pip from pulling a different torch/CUDA version.
safe_install("transformers>=4.49 accelerate>=1.0", "--no-deps")
safe_install("bitsandbytes sentencepiece tiktoken einops timm")
safe_install("fastapi uvicorn pyngrok nest-asyncio Pillow")

import torch
print("=" * 50)
print(f"✅ Dependencies ready!")
print(f"   PyTorch:  {torch.__version__}")
print(f"   CUDA:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        c = torch.cuda.get_device_capability(i)
        print(f"   GPU {i}: {p.name} ({p.total_memory / 1e9:.1f} GB, compute {c[0]}.{c[1]})")
print("=" * 50)

try:
    import transformers, accelerate, tokenizers
    print(f"   transformers: {transformers.__version__}")
    print(f"   accelerate:   {accelerate.__version__}")
    print(f"   tokenizers:   {tokenizers.__version__}")
except ImportError as e:
    print(f"❌ MISSING: {e}  — Enable Internet in Kaggle Settings and restart.")

## 2. Load CogAgent-9B Model

- Official model weights are **BF16** — T4 GPUs (compute 7.5) can't handle BF16 natively
- Loading BF16 weights into FP16 causes overflow → NaN → garbage output
- **Fix: INT4 (NF4) quantization** via `bitsandbytes` — officially supported in CogAgent's `cli_demo.py`
- Model shrinks from ~18GB to ~5GB, fits easily on T4 x2
- Uses `device_map="auto"` + `tokenizer.apply_chat_template()` (official API)
- DynamicCache→tuple patches for transformers ≥4.49 compatibility


In [ ]:
import os, sys, gc, traceback
import torch
import numpy as np

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

MODEL_ID = "THUDM/cogagent-9b-20241220"

# ═══════════════════════════════════════════════════════════════
# WHY INT4 QUANTIZATION + BF16 COMPUTE IS REQUIRED ON T4
# ═══════════════════════════════════════════════════════════════
# The model weights are stored in BF16 (range up to 3.4e38).
# Official cli_demo.py loads with torch_dtype=torch.bfloat16.
# T4 (compute 7.5) has NO native BF16 tensor cores.
#
# Problem 1: BF16 weights -> FP16 causes overflow (max 65504) -> Inf -> NaN
# Problem 2: Even with INT4 weights, FP16 compute overflows in attention/MLP
#
# FIX: INT4 (NF4) quantization + BF16 compute dtype.
#   quantization_config=BitsAndBytesConfig(load_in_4bit=True)
#   bnb_4bit_compute_dtype=torch.bfloat16
# T4 supports BF16 tensors - PyTorch upcasts to FP32 internally.
# No tensor-core speedup, but math stays correct (no overflow).
# ═══════════════════════════════════════════════════════════════

if torch.cuda.is_available():
    _cap = torch.cuda.get_device_capability(0)
    print(f"GPU compute capability {_cap[0]}.{_cap[1]}")
    if _cap[0] >= 8:
        COMPUTE_DTYPE = torch.bfloat16
        NEEDS_QUANT = False
    else:
        COMPUTE_DTYPE = torch.bfloat16
        NEEDS_QUANT = True
        print("   Using BF16 compute on T4 (emulated via FP32, no overflow)")
else:
    COMPUTE_DTYPE = torch.float32
    NEEDS_QUANT = False

print(f"Compute dtype: {COMPUTE_DTYPE}, quantization: {'INT4 NF4' if NEEDS_QUANT else 'none'}")
print(f"\nLoading {MODEL_ID} ...")

# ═══════════════════════════════════════════════════════════════
# Helper: DynamicCache -> tuple-of-tuples (ChatGLM native format)
# ═══════════════════════════════════════════════════════════════
def _dyncache_to_tuples(cache):
    if cache is None:
        return None
    if isinstance(cache, (tuple, list)):
        return cache
    if hasattr(cache, 'key_cache') and hasattr(cache, 'value_cache'):
        if len(cache.key_cache) == 0:
            return None
        return tuple(
            (k, v) for k, v in zip(cache.key_cache, cache.value_cache)
        )
    return None

# ═══════════════════════════════════════════════════════════════
# Compatibility patches (run BEFORE from_pretrained)
# ═══════════════════════════════════════════════════════════════

# -- Patch 1: _tied_weights_keys guard for accelerate device map --
try:
    import transformers.integrations.accelerate as _accel_mod
    _orig_init_infer = _accel_mod._init_infer_auto_device_map
    def _patched_init_infer(model, *args, **kwargs):
        if not hasattr(model, '_tied_weights_keys') or model._tied_weights_keys is None:
            model._tied_weights_keys = []
        if not hasattr(model, 'all_tied_weights_keys'):
            model.all_tied_weights_keys = {}
        return _orig_init_infer(model, *args, **kwargs)
    _accel_mod._init_infer_auto_device_map = _patched_init_infer
    print("   [patch 1] _tied_weights_keys guard -> OK")
except Exception as e:
    print(f"   [patch 1] skipped ({e})")

# -- Patch 2: Lenient check_imports (skip optional xformers / triton) --
try:
    from transformers.utils import import_utils as _iu
    _orig_check = _iu.check_imports
    def _lenient_check(filename):
        try:
            return _orig_check(filename)
        except ImportError as e:
            if any(m in str(e) for m in ('xformers', 'triton', 'flash_attn', 'liger_kernel')):
                return []
            raise
    _iu.check_imports = _lenient_check
    print("   [patch 2] lenient check_imports -> OK")
except Exception:
    print("   [patch 2] skipped")

# -- Patch 8: Guard get_keys_to_not_convert for BitsAndBytes quantizer --
try:
    from transformers.quantizers import base as _quant_base
    _orig_get_keys_to_not_convert = _quant_base.get_keys_to_not_convert
    def _safe_get_keys_to_not_convert(model):
        if not hasattr(model, 'all_tied_weights_keys'):
            model.all_tied_weights_keys = {}
        if not hasattr(model, '_tied_weights_keys') or model._tied_weights_keys is None:
            model._tied_weights_keys = []
        return _orig_get_keys_to_not_convert(model)
    _quant_base.get_keys_to_not_convert = _safe_get_keys_to_not_convert
    print("   [patch 8] get_keys_to_not_convert guard -> OK")
except Exception as e:
    print(f"   [patch 8] skipped ({e})")

# -- Patch 9: Allow BnB 4-bit quantization with CPU offload --
# BnB's validate_environment() unconditionally rejects explicit device_map
# dicts that map ANY module to CPU/disk.  But we NEED CPU offload because:
#   - The vision encoder (~4B params) is NOT quantized by BnB (only nn.Linear
#     layers in the GLM blocks are).  It stays in full BF16 -> ~8 GiB.
#   - Two T4s (14.56 GiB each) can't hold 8 GiB vision + 5 GiB INT4 LM
#     simultaneously during the loading transient.
# Fix: monkey-patch validate_environment to suppress this one error.
# CPU-offloaded modules stay in BF16 (unquantized) -- correct for inference.
try:
    from transformers.quantizers.quantizer_bnb_4bit import Bnb4BitHfQuantizer
    _orig_bnb4_validate = Bnb4BitHfQuantizer.validate_environment
    def _patched_bnb4_validate(self, *args, **kwargs):
        try:
            return _orig_bnb4_validate(self, *args, **kwargs)
        except ValueError as e:
            if "Some modules are dispatched on the CPU or the disk" in str(e):
                print("   [patch 9] Allowing CPU offload for unquantized modules")
                return
            raise
    Bnb4BitHfQuantizer.validate_environment = _patched_bnb4_validate
    print("   [patch 9] BnB 4-bit CPU offload guard -> OK")
except Exception as e:
    print(f"   [patch 9] skipped ({e})")

# -- Patch 10: Fix Params4bit rejecting _is_hf_initialized kwarg --
# When accelerate dispatches CPU-offloaded BnB 4-bit modules,
# set_module_tensor_to_device() rebuilds parameters via:
#     param_cls(new_value, requires_grad=..., **kwargs)
# where kwargs = old_value.__dict__ which may contain '_is_hf_initialized'
# (injected by transformers).  Params4bit.__new__() doesn't accept it.
#
# Fix: Monkey-patch Params4bit.__new__ to pop unknown kwargs before calling
# the original.  This is the only reliable interception point because the
# kwargs dict is built inside set_module_tensor_to_device as a local var.
try:
    import bitsandbytes as bnb
    _Orig4bit = bnb.nn.Params4bit
    _orig_4bit_new = _Orig4bit.__new__

    # Also patch Int8Params if present (same issue can occur with 8-bit)
    _has_int8 = hasattr(bnb.nn, 'Int8Params')

    # Params4bit.__new__ is a staticmethod that takes cls as first arg
    def _safe_params4bit_new(cls, *args, **kwargs):
        kwargs.pop('_is_hf_initialized', None)
        kwargs.pop('_is_quantized', None)
        return _orig_4bit_new(cls, *args, **kwargs)
    bnb.nn.Params4bit.__new__ = _safe_params4bit_new

    if _has_int8:
        _orig_int8_new = bnb.nn.Int8Params.__new__
        def _safe_int8params_new(cls, *args, **kwargs):
            kwargs.pop('_is_hf_initialized', None)
            kwargs.pop('_is_quantized', None)
            return _orig_int8_new(cls, *args, **kwargs)
        bnb.nn.Int8Params.__new__ = _safe_int8params_new

    print("   [patch 10] Params4bit.__new__ _is_hf_initialized fix -> OK")
except Exception as e:
    print(f"   [patch 10] skipped ({e})")

# -- Patch 11: Fix Linear4bit state_dict crash on meta tensors --
# During dispatch, accelerate moves params to meta device via
# set_module_tensor_to_device(module, name, "meta") (hooks.py:308).
# Then attach_execution_device_hook (hooks.py:428) calls
#     len(module.state_dict()) > 0
# just to check if method has params.  For BnB Linear4bit, this triggers
# _save_to_state_dict -> quant_state.as_dict -> offset.item()
# which crashes because offset is now a meta tensor.
# Fix: patch _save_to_state_dict to catch the meta tensor error.
try:
    from bitsandbytes.nn.modules import Linear4bit as _L4b
    _orig_l4b_save = _L4b._save_to_state_dict
    def _safe_l4b_save(self, destination, prefix, keep_vars):
        try:
            return _orig_l4b_save(self, destination, prefix, keep_vars)
        except RuntimeError as e:
            if "meta" in str(e).lower():
                # Fallback: save raw weight without quant_state serialization
                # This is fine -- accelerate only calls state_dict() for a
                # length check during dispatch, not for actual saving.
                import torch
                torch.nn.Module._save_to_state_dict(self, destination, prefix, keep_vars)
                return
            raise
    _L4b._save_to_state_dict = _safe_l4b_save
    print("   [patch 11] Linear4bit state_dict meta-tensor guard -> OK")
except Exception as e:
    print(f"   [patch 11] skipped ({e})")

# -- Patch 12: Guard caching_allocator_warmup for all_tied_weights_keys --
# transformers' get_total_byte_count() (called during model loading) does:
#   tied_param_names = model.all_tied_weights_keys.keys()
# ChatGLM models don't have this attribute, causing AttributeError.
# Fix: wrap caching_allocator_warmup to ensure the attribute exists.
try:
    import transformers.modeling_utils as _mu
    if hasattr(_mu, 'caching_allocator_warmup'):
        _orig_caw = _mu.caching_allocator_warmup
        def _safe_caw(model, *args, **kwargs):
            if not hasattr(model, 'all_tied_weights_keys'):
                model.all_tied_weights_keys = {}
            if not hasattr(model, '_tied_weights_keys') or model._tied_weights_keys is None:
                model._tied_weights_keys = []
            return _orig_caw(model, *args, **kwargs)
        _mu.caching_allocator_warmup = _safe_caw
        print("   [patch 12] caching_allocator_warmup guard -> OK")
    elif hasattr(_mu, 'get_total_byte_count'):
        _orig_gtbc = _mu.get_total_byte_count
        def _safe_gtbc(model, *args, **kwargs):
            if not hasattr(model, 'all_tied_weights_keys'):
                model.all_tied_weights_keys = {}
            if not hasattr(model, '_tied_weights_keys') or model._tied_weights_keys is None:
                model._tied_weights_keys = []
            return _orig_gtbc(model, *args, **kwargs)
        _mu.get_total_byte_count = _safe_gtbc
        print("   [patch 12] get_total_byte_count guard -> OK")
    else:
        print("   [patch 12] skipped (no caching_allocator_warmup or get_total_byte_count)")
except Exception as e:
    print(f"   [patch 12] skipped ({e})")

# -- Patch 13: Fix replace_with_bnb_linear to skip entire vision subtree --
# llm_int8_skip_modules=["vision"] adds bare "vision" to the skip list.
# But replace_with_bnb_linear checks FULL dotted paths ("transformer.vision").
# "transformer.vision" != "vision" -> vision's Linear layers still get quantized
# to Linear4bit -> BnB quant_state on meta device -> crash during offload restore.
# Fix: expand bare skip names into full dotted paths.
try:
    _replace_fn_orig = None
    _replace_locations = []

    try:
        import transformers.integrations.bitsandbytes as _bnb_integ
        if hasattr(_bnb_integ, 'replace_with_bnb_linear'):
            _replace_fn_orig = _bnb_integ.replace_with_bnb_linear
            _replace_locations.append((_bnb_integ, 'replace_with_bnb_linear'))
    except ImportError:
        pass

    try:
        import transformers.integrations as _integ
        if hasattr(_integ, 'replace_with_bnb_linear'):
            if _replace_fn_orig is None:
                _replace_fn_orig = _integ.replace_with_bnb_linear
            _replace_locations.append((_integ, 'replace_with_bnb_linear'))
    except ImportError:
        pass

    try:
        import transformers.quantizers.quantizer_bnb_4bit as _q4mod
        if hasattr(_q4mod, 'replace_with_bnb_linear'):
            if _replace_fn_orig is None:
                _replace_fn_orig = _q4mod.replace_with_bnb_linear
            _replace_locations.append((_q4mod, 'replace_with_bnb_linear'))
    except ImportError:
        pass

    if _replace_fn_orig is not None:
        _p13_orig = _replace_fn_orig

        def _vision_skip_replace(model, modules_to_not_convert=None, **kw):
            # Only expand on the top-level call (no current_key_name yet)
            if kw.get('current_key_name') is None and 'current_key_name' not in kw:
                if modules_to_not_convert is None:
                    modules_to_not_convert = []
                _skip_bare = set(modules_to_not_convert)
                _expanded = set(modules_to_not_convert)

                for full_name, _ in model.named_modules():
                    parts = full_name.split('.')
                    for skip_name in _skip_bare:
                        if skip_name in parts:
                            idx = parts.index(skip_name)
                            _expanded.add('.'.join(parts[:idx + 1]))

                _new_entries = _expanded - _skip_bare
                if _new_entries:
                    print(f"   [patch 13] Expanded skip list with: {_new_entries}")
                modules_to_not_convert = list(_expanded)

            return _p13_orig(
                model,
                modules_to_not_convert=modules_to_not_convert,
                **kw,
            )

        for _mod, _attr in _replace_locations:
            setattr(_mod, _attr, _vision_skip_replace)

        print(f"   [patch 13] replace_with_bnb_linear vision skip -> OK ({len(_replace_locations)} locations)")
    else:
        print("   [patch 13] skipped (replace_with_bnb_linear not found)")
except Exception as e:
    print(f"   [patch 13] skipped ({e})")

# -- Patch 3: ChatGLM config -> standard HF attribute aliases --
def _patch_chatglm_config(cfg):
    if not hasattr(cfg, 'max_length') or getattr(cfg, 'max_length', None) is None:
        cfg.max_length = getattr(cfg, 'seq_length', 8192)
    for hf, glm in [('num_hidden_layers', 'num_layers'),
                     ('max_position_embeddings', 'seq_length'),
                     ('head_dim', 'kv_channels')]:
        if not hasattr(cfg, hf) and hasattr(cfg, glm):
            setattr(cfg, hf, getattr(cfg, glm))
    if not hasattr(cfg, 'num_key_value_heads'):
        if hasattr(cfg, 'multi_query_group_num'):
            cfg.num_key_value_heads = cfg.multi_query_group_num
        elif hasattr(cfg, 'num_attention_heads'):
            cfg.num_key_value_heads = cfg.num_attention_heads
    for k, v in {'is_encoder_decoder': False, 'tie_word_embeddings': False,
                  'use_cache': True, 'return_dict': True}.items():
        if not hasattr(cfg, k):
            setattr(cfg, k, v)
    for gk in ('output_scores', 'return_dict_in_generate'):
        cfg.__dict__.pop(gk, None)
print("   [patch 3] ChatGLMConfig aliases ready")

# -- Patch 4: _prepare_generation_config guard --
from transformers import GenerationMixin as _GM, GenerationConfig as _GC
if hasattr(_GM, '_prepare_generation_config'):
    _orig_pgc = _GM._prepare_generation_config
    def _safe_pgc(self, gen_cfg, *a, **kw):
        try:
            return _orig_pgc(self, gen_cfg, *a, **kw)
        except (ValueError, AttributeError):
            if gen_cfg is None:
                gen_cfg = _GC.from_model_config(self.config) if hasattr(self, 'config') else _GC()
            gen_cfg.max_length = getattr(gen_cfg, 'max_length', 8192) or 8192
            gen_cfg.max_new_tokens = kw.get('max_new_tokens', None)
            for _gk in ('output_scores', 'return_dict_in_generate', 'max_length',
                         'max_new_tokens', 'min_length', 'do_sample', 'temperature',
                         'top_k', 'top_p', 'repetition_penalty'):
                if hasattr(self, 'config'):
                    self.config.__dict__.pop(_gk, None)
            for k2, v2 in kw.items():
                if hasattr(gen_cfg, k2):
                    setattr(gen_cfg, k2, v2)
            return gen_cfg, kw
    _GM._prepare_generation_config = _safe_pgc
    print("   [patch 4] _prepare_generation_config guard -> OK")

# -- Patch 5: Fix _extract_past_from_model_output --
_has_extract = hasattr(_GM, '_extract_past_from_model_output')
_is_static = isinstance(
    _GM.__dict__.get('_extract_past_from_model_output'), staticmethod
) if _has_extract else False

if _has_extract and _is_static:
    _orig_extract = _GM._extract_past_from_model_output.__func__
    def _compat_extract_s(outputs, **kwargs):
        try:
            result = _orig_extract(outputs, **kwargs)
        except (TypeError, AttributeError):
            result = ("past_key_values", getattr(outputs, 'past_key_values', None))
        if isinstance(result, tuple) and len(result) == 2:
            cache_name, cache = result
        else:
            cache_name, cache = "past_key_values", result
        cache = _dyncache_to_tuples(cache)
        return cache_name, cache
    _GM._extract_past_from_model_output = staticmethod(_compat_extract_s)
    print("   [patch 5] _extract_past -> DynamicCache->tuples (static) -> OK")
elif _has_extract:
    _orig_extract_i = _GM._extract_past_from_model_output
    def _compat_extract_i(self, outputs, **kwargs):
        try:
            result = _orig_extract_i(self, outputs, **kwargs)
        except (TypeError, AttributeError):
            result = ("past_key_values", getattr(outputs, 'past_key_values', None))
        if isinstance(result, tuple) and len(result) == 2:
            cache_name, cache = result
        else:
            cache_name, cache = "past_key_values", result
        cache = _dyncache_to_tuples(cache)
        return cache_name, cache
    _GM._extract_past_from_model_output = _compat_extract_i
    print("   [patch 5] _extract_past -> DynamicCache->tuples (instance) -> OK")
else:
    def _new_extract(self, outputs, **kwargs):
        cache = getattr(outputs, 'past_key_values', None)
        cache = _dyncache_to_tuples(cache)
        return "past_key_values", cache
    _GM._extract_past_from_model_output = _new_extract
    print("   [patch 5] _extract_past -> DynamicCache->tuples (new shim) -> OK")

# -- Patch 6: Convert DynamicCache in _prepare_cache_for_generation --
if hasattr(_GM, '_prepare_cache_for_generation'):
    _orig_pcfg = _GM._prepare_cache_for_generation
    def _tuples_pcfg(self, *args, **kwargs):
        result = _orig_pcfg(self, *args, **kwargs)
        if isinstance(result, dict) and 'past_key_values' in result:
            result['past_key_values'] = _dyncache_to_tuples(result['past_key_values'])
        for a in args:
            if isinstance(a, dict) and 'past_key_values' in a:
                a['past_key_values'] = _dyncache_to_tuples(a['past_key_values'])
        return result
    _GM._prepare_cache_for_generation = _tuples_pcfg
    print("   [patch 6] _prepare_cache -> DynamicCache->tuples -> OK")

# ═══════════════════════════════════════════════════════════════
# InvalidScoreLogitsProcessor -- clamps NaN/Inf logits + diagnostic
# ═══════════════════════════════════════════════════════════════
from transformers import LogitsProcessor

class InvalidScoreLogitsProcessor(LogitsProcessor):
    def __init__(self):
        self._nan_count = 0
    def __call__(self, input_ids, scores):
        if torch.isnan(scores).any() or torch.isinf(scores).any():
            self._nan_count += 1
            if self._nan_count == 1:
                pct = (torch.isnan(scores) | torch.isinf(scores)).float().mean().item() * 100
                print(f"   WARNING: NaN/Inf logits on step 1! ({pct:.0f}% of vocab)")
            scores.zero_()
            scores[..., 198] = 5e4
        return scores

# ═══════════════════════════════════════════════════════════════
# Load model with INT4 NF4 quantization (official option from cli_demo.py)
# ═══════════════════════════════════════════════════════════════
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig, BitsAndBytesConfig
from PIL import Image

config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
_patch_chatglm_config(config)

# -- Pre-clear GPU memory before loading --
gc.collect()
for _gi in range(torch.cuda.device_count()):
    with torch.cuda.device(_gi):
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

# -----------------------------------------------------------------
# Pre-compute device_map at BF16 scale, then pass explicit dict.
#
# WHY: BnB only quantizes nn.Linear layers in the LM.  The vision
# encoder (~4B params) stays in full BF16 (~8 GiB).  But accelerate's
# auto planner estimates ALL layers at INT4 size (~0.5 B/param),
# packing vision + LM onto GPUs.  During loading the BF16 vision
# weights overflow the GPU.
#
# FIX: estimate sizes at BF16 (2 B/param) so the planner correctly
# puts the vision encoder on CPU.  Patch 9 allows the explicit dict
# through BnB's validator.
# -----------------------------------------------------------------
from accelerate import infer_auto_device_map, init_empty_weights

_max_memory = {}
for _gi in range(torch.cuda.device_count()):
    _total_gib = torch.cuda.get_device_properties(_gi).total_memory / (1024**3)
    _limit = int(_total_gib - 2)  # 2 GiB headroom
    _max_memory[_gi] = f"{_limit}GiB"
_max_memory["cpu"] = "24GiB"

_offload_dir = "/tmp/cogagent_offload"
os.makedirs(_offload_dir, exist_ok=True)

print(f"   Computing device map (BF16 size estimates)...")
try:
    with init_empty_weights():
        _empty_model = AutoModelForCausalLM.from_config(config, trust_remote_code=True)
    _no_split = getattr(_empty_model, '_no_split_modules', None) or []
    _device_map = infer_auto_device_map(
        _empty_model,
        max_memory=_max_memory,
        dtype=torch.bfloat16,
        no_split_module_classes=_no_split,
    )
    del _empty_model
    gc.collect()
    print(f"   Device map computed with BF16 estimates")
except Exception as _dm_err:
    print(f"   WARNING: infer_auto_device_map failed ({_dm_err}), using manual map")
    _device_map = {"transformer.embedding": 0, "transformer.rotary_pos_emb": 0}
    for _li in range(40):
        _device_map[f"transformer.encoder.layers.{_li}"] = 0 if _li < 20 else 1
    _device_map["transformer.encoder.final_layernorm"] = 1
    _device_map["transformer.output_layer"] = 0
    _device_map["transformer.vision"] = "cpu"

_dev_summary = {}
for _v in _device_map.values():
    _dev_summary[str(_v)] = _dev_summary.get(str(_v), 0) + 1
print(f"   device_map layers: {_dev_summary}")
print(f"   max_memory: {_max_memory}, offload: {_offload_dir}")

if NEEDS_QUANT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        llm_int8_skip_modules=["vision"],  # keep vision in BF16 (CPU offload safe)
    )
    print(f"   INT4 NF4 quantization, compute dtype: {COMPUTE_DTYPE}")
    print(f"   Vision encoder excluded from quantization (stays BF16 for CPU offload)")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        config=config,
        quantization_config=quantization_config,
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
        device_map=_device_map,
        offload_folder=_offload_dir,
        offload_state_dict=True,
        low_cpu_mem_usage=True,
    ).eval()
else:
    print(f"   Loading in {COMPUTE_DTYPE} (no quantization needed)")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        config=config,
        torch_dtype=COMPUTE_DTYPE,
        trust_remote_code=True,
        device_map=_device_map,
        offload_folder=_offload_dir,
        low_cpu_mem_usage=True,
    ).eval()

print(f"\nModel {MODEL_ID} loaded!")
print(f"   Model class: {type(model).__name__}")
print(f"   Quantized: {NEEDS_QUANT}")

gc.collect(); torch.cuda.empty_cache()
_n = torch.cuda.device_count()
for _i in range(_n):
    _used = torch.cuda.memory_allocated(_i) / 1e9
    _total = torch.cuda.get_device_properties(_i).total_memory / 1e9
    print(f"   GPU {_i}: {_used:.1f} / {_total:.1f} GiB ({_total - _used:.1f} GiB free)")

_meta_count = sum(1 for p in model.parameters() if p.device.type == 'meta')
if _meta_count:
    print(f"   WARNING: {_meta_count} params on meta-device")
else:
    print(f"   All model parameters on GPU/CPU")

# ═══════════════════════════════════════════════════════════════
# Post-load: config + generation_config cleanup
# ═══════════════════════════════════════════════════════════════
_patch_chatglm_config(model.config)
for _gp in ('output_scores', 'return_dict_in_generate',
            'max_new_tokens', 'min_length', 'do_sample', 'temperature',
            'top_k', 'top_p', 'repetition_penalty'):
    model.config.__dict__.pop(_gp, None)

_max_len = getattr(model.config, 'seq_length', 8192)
model.generation_config.max_length = _max_len
model.generation_config.max_new_tokens = None
model.generation_config.cache_implementation = None
if hasattr(model.generation_config, 'cache_config'):
    model.generation_config.cache_config = None
print(f"   generation_config.max_length = {model.generation_config.max_length}")

# -- Post-load patch A: transformer.forward -- DynamicCache -> tuples --
_TfCls = type(model.transformer)
_orig_tf_fwd = _TfCls.forward
def _tf_fwd_compat_cache(self, *args, **kwargs):
    pkv = kwargs.get('past_key_values', None)
    if pkv is not None and not isinstance(pkv, (tuple, list)):
        kwargs['past_key_values'] = _dyncache_to_tuples(pkv)
    return _orig_tf_fwd(self, *args, **kwargs)
_TfCls.forward = _tf_fwd_compat_cache
print("   [fix-A] transformer.forward DynamicCache->tuples wrapper")

# -- Post-load patch B: prepare_inputs_for_generation -- DynamicCache -> tuples --
_ModelCls = type(model)
if hasattr(_ModelCls, 'prepare_inputs_for_generation'):
    _orig_pig = _ModelCls.prepare_inputs_for_generation
    def _patched_pig(self, input_ids, past_key_values=None, **kwargs):
        past_key_values = _dyncache_to_tuples(past_key_values)
        return _orig_pig(self, input_ids, past_key_values=past_key_values, **kwargs)
    _ModelCls.prepare_inputs_for_generation = _patched_pig
    print("   [fix-B] prepare_inputs_for_generation DynamicCache->tuples wrapper")

# ═══════════════════════════════════════════════════════════════
# Tokenizer
# ═══════════════════════════════════════════════════════════════
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
print(f"   tokenizer ready (vocab={tokenizer.vocab_size})")

# -- Patch 7: Fix missing batch_encode_plus in transformers>=4.49 --
_TokCls = type(tokenizer)
if not hasattr(tokenizer, 'batch_encode_plus'):
    from transformers.tokenization_utils_base import BatchEncoding as _BE
    def _batch_encode_plus(self, batch, padding=False, truncation=False,
                           max_length=None, return_tensors=None,
                           is_split_into_words=False, add_special_tokens=True, **kw):
        if isinstance(batch[0], int):
            batch = [batch]
        ids = [list(seq) for seq in batch]
        mask = [[1] * len(seq) for seq in ids]
        data = {"input_ids": ids, "attention_mask": mask}
        if return_tensors == "pt":
            data = {k: torch.tensor(v) for k, v in data.items()}
        return _BE(data)
    _TokCls.batch_encode_plus = _batch_encode_plus
    print("   [patch 7] batch_encode_plus shim -> OK")
else:
    print("   [patch 7] batch_encode_plus already exists -> skip")

INPUT_DEVICE = torch.device('cuda:0')
print(f"   input device: {INPUT_DEVICE}")

# -- Fix-C: FP16 vision + memory-efficient SDPA on GPU --
# OOM root cause: BF16 tensors → PyTorch SDPA uses "math" backend on T4 →
# materializes full 6400×6400×16 attention matrix = 2.44 GiB → OOM.
# T4's memory-efficient SDPA backend only works with FP16, not BF16.
# Fix: convert vision to FP16 → enables mem-efficient SDPA → attention
# memory drops from 2.44 GiB to ~50 MB.
# Dynamic offload: vision rests on CPU, moves to GPU for forward, back to CPU.
try:
    from accelerate.hooks import remove_hook_from_module
    import time as _fc_time

    # 1. Remove ALL accelerate hooks from vision submodules
    _fix_c_removed = 0
    for _vname, _vmod in model.transformer.vision.named_modules():
        if hasattr(_vmod, '_hf_hook'):
            remove_hook_from_module(_vmod, recurse=False)
            _fix_c_removed += 1

    # 2. Convert vision to FP16 (NOT BF16!) and keep on CPU at rest
    # FP16 is critical: it enables memory-efficient SDPA on T4 (compute 7.5)
    # BF16 forces math-mode SDPA = 2.44 GiB attention matrix = OOM
    model.transformer.vision = model.transformer.vision.to(
        dtype=torch.float16, device=torch.device('cpu')
    )
    _v_total = sum(1 for _ in model.transformer.vision.parameters())
    _v_fp16 = sum(1 for p in model.transformer.vision.parameters() if p.dtype == torch.float16)
    print(f"   [fix-C] Removed {_fix_c_removed} hooks, {_v_fp16}/{_v_total} vision params converted to FP16")

    # 3. Ensure memory-efficient SDPA is enabled
    if hasattr(torch.backends.cuda, 'enable_mem_efficient_sdp'):
        torch.backends.cuda.enable_mem_efficient_sdp(True)
        print(f"   [fix-C] Memory-efficient SDPA enabled")

    # 4. Pick the GPU with the most free memory for vision forward passes
    _vision_gpu = torch.device('cuda:0')
    _best_free = 0
    for _gi in range(torch.cuda.device_count()):
        _free = torch.cuda.get_device_properties(_gi).total_memory - torch.cuda.memory_allocated(_gi)
        if _free > _best_free:
            _best_free = _free
            _vision_gpu = torch.device(f'cuda:{_gi}')

    # 5. Find LM target device (where token embeddings live)
    _lm_target_device = torch.device('cuda:0')
    for _pname, _pparam in model.named_parameters():
        if 'encoder.layers.0.' in _pname and _pparam.device.type == 'cuda':
            _lm_target_device = _pparam.device
            break

    _vision_bytes = sum(p.numel() * p.element_size() for p in model.transformer.vision.parameters())
    print(f"   [fix-C] Vision: {_vision_bytes/1e9:.1f} GB FP16 weights")
    print(f"   [fix-C] Will run on {_vision_gpu} ({_best_free/1e9:.1f} GB free), output -> {_lm_target_device}")

    # 6. Wrap vision.forward: CPU->GPU, FP16 autocast, GPU->CPU
    _orig_vision_fwd = model.transformer.vision.forward

    def _vision_dynamic_gpu(*args, **kwargs):
        _t0 = _fc_time.time()

        # Move vision weights to GPU
        model.transformer.vision.to(_vision_gpu)

        # Move inputs to GPU as FP16
        new_args = tuple(
            a.to(device=_vision_gpu, dtype=torch.float16) if isinstance(a, torch.Tensor) else a
            for a in args
        )
        new_kwargs = {
            k: (v.to(device=_vision_gpu, dtype=torch.float16) if isinstance(v, torch.Tensor) else v)
            for k, v in kwargs.items()
        }

        # Run with FP16 autocast → forces memory-efficient SDPA on T4
        with torch.no_grad(), torch.autocast(device_type='cuda', dtype=torch.float16):
            result = _orig_vision_fwd(*new_args, **new_kwargs)

        # Move output to LM device as BF16 (LM expects BF16)
        if isinstance(result, torch.Tensor):
            result = result.to(device=_lm_target_device, dtype=torch.bfloat16)

        # Move vision back to CPU to free GPU for text decoding KV cache
        model.transformer.vision.to(torch.device('cpu'))
        torch.cuda.empty_cache()

        _t1 = _fc_time.time()
        print(f"   [fix-C] Vision forward: {_t1-_t0:.1f}s on {_vision_gpu}")
        return result

    model.transformer.vision.forward = _vision_dynamic_gpu
    print(f"   [fix-C] Dynamic FP16 GPU offload ready")

    gc.collect(); torch.cuda.empty_cache()
    for _gi in range(torch.cuda.device_count()):
        _used = torch.cuda.memory_allocated(_gi) / 1e9
        _total = torch.cuda.get_device_properties(_gi).total_memory / 1e9
        print(f"   [fix-C] GPU {_gi}: {_used:.1f} / {_total:.1f} GiB (vision at rest on CPU)")
except Exception as _fc_err:
    print(f"   [fix-C] FAILED ({_fc_err})")
    import traceback; traceback.print_exc()

# ═══════════════════════════════════════════════════════════════
# Generation config -- matches official cli_demo.py:
#   gen_kwargs = {"max_length": 4096, "do_sample": True, "top_k": 1}
# ═══════════════════════════════════════════════════════════════
from transformers import GenerationConfig

GEN_CONFIG = GenerationConfig(
    max_length=4096,
    do_sample=True,
    top_k=1,
)

WARMUP_GEN_CONFIG = GenerationConfig(
    max_new_tokens=50,
    do_sample=True,
    top_k=1,
)

TEST_GEN_CONFIG = GenerationConfig(
    max_new_tokens=200,
    do_sample=True,
    top_k=1,
)

GEN_EXTRA = dict(
    logits_processor=[InvalidScoreLogitsProcessor()],
)
print(f"   GenerationConfig: max_length={GEN_CONFIG.max_length}, do_sample={GEN_CONFIG.do_sample}, top_k={GEN_CONFIG.top_k}")
print(f"   Warm-up config:   max_new_tokens={WARMUP_GEN_CONFIG.max_new_tokens}")

# ═══════════════════════════════════════════════════════════════
# Warm-up tests
# ═══════════════════════════════════════════════════════════════
import time as _time

print("\n-- Warm-up test 1 (text-only) --")
try:
    _inp = tokenizer.apply_chat_template(
        [{"role": "user", "content": "What is 2+2? Answer with just the number."}],
        add_generation_prompt=True, tokenize=True,
        return_tensors="pt", return_dict=True,
    ).to(INPUT_DEVICE)
    print(f"   input keys: {list(_inp.keys())}, shape: {_inp['input_ids'].shape}")

    _t0 = _time.time()
    with torch.no_grad():
        _out = model.generate(**_inp, generation_config=WARMUP_GEN_CONFIG, **GEN_EXTRA)
    _t1 = _time.time()
    _new = _out[0, _inp['input_ids'].shape[1]:]
    _text = tokenizer.decode(_new, skip_special_tokens=True).strip()
    print(f"   Output ({len(_new)} tokens, {_t1-_t0:.1f}s): {_text[:200]}")
    _lp = GEN_EXTRA['logits_processor'][0]
    if _lp._nan_count > 0:
        print(f"   WARNING: NaN/Inf on {_lp._nan_count}/{len(_new)} steps")
    elif any(c.isalnum() for c in _text):
        print("   Text warm-up passed!")
    else:
        print("   Output may be garbled (but no NaN)")
except Exception as e:
    print(f"   Text warm-up error: {e}")
    traceback.print_exc()

gc.collect(); torch.cuda.empty_cache()
GEN_EXTRA['logits_processor'] = [InvalidScoreLogitsProcessor()]

print("\n-- Warm-up test 2 (image) --")
try:
    _tiny = Image.new("RGB", (64, 64), (255, 0, 0))
    _inp2 = tokenizer.apply_chat_template(
        [{"role": "user", "image": _tiny, "content": "Describe this image briefly."}],
        add_generation_prompt=True, tokenize=True,
        return_tensors="pt", return_dict=True,
    ).to(INPUT_DEVICE)
    print(f"   input keys: {list(_inp2.keys())}, shape: {_inp2['input_ids'].shape}")

    _t0 = _time.time()
    with torch.no_grad():
        _out2 = model.generate(**_inp2, generation_config=WARMUP_GEN_CONFIG, **GEN_EXTRA)
    _t1 = _time.time()
    _new2 = _out2[0, _inp2['input_ids'].shape[1]:]
    _text2 = tokenizer.decode(_new2, skip_special_tokens=True).strip()
    print(f"   Output ({len(_new2)} tokens, {_t1-_t0:.1f}s): {_text2[:200]}")
    _lp2 = GEN_EXTRA['logits_processor'][0]
    if _lp2._nan_count > 0:
        print(f"   WARNING: NaN/Inf on {_lp2._nan_count}/{len(_new2)} steps")
    elif any(c.isalnum() for c in _text2):
        print("   Vision warm-up passed!")
    else:
        print("   Output may be garbled (but no NaN)")
except Exception as e:
    print(f"   Vision warm-up error: {e}")
    traceback.print_exc()

GEN_EXTRA['logits_processor'] = [InvalidScoreLogitsProcessor()]
gc.collect(); torch.cuda.empty_cache()
print("\nAll patches applied, model + tokenizer ready.")

## 3. Define Vision API Prompt & Inference
Uses `tokenizer.apply_chat_template()` — the same API as the official `inference/cli_demo.py`.


In [ ]:
# ============================================================
# Cell 3: Vision Inference Function
# ============================================================
import base64, json, re, io, time
from PIL import Image

# ═══════════════════════════════════════════════════════════════
#  CogAgent output parser — converts model text → action JSON
# ═══════════════════════════════════════════════════════════════

def _parse_cogagent_native(text, img_w=1920, img_h=1080):
    """Parse CogAgent's native grounding format -> action JSON.

    CogAgent Dec 2024 format:
      Status: ...
      Action: ...
      Grounded Operation: CLICK(box=[[x1,y1,x2,y2]], element_info='...')
      (coordinates in 0-1000 system)
    """
    text = text.strip()

    grounded = re.search(r"Grounded Operation:\s*(.*?)(?:\n|$)", text)
    action_m = re.search(r"Action:\s*(.*?)(?:\n|$)", text)
    status_m = re.search(r"Status:\s*(.*?)(?:\n|$)", text)

    step = grounded.group(1).strip() if grounded else None
    action_desc = action_m.group(1).strip() if action_m else ""
    status = status_m.group(1).strip().lower() if status_m else ""

    if status in ("finish", "finished", "done", "completed"):
        return {"action": "done", "description": action_desc or "Task completed"}

    if step:
        op_match = re.match(r'(\w+)\s*\(', step)
        if op_match:
            operation = op_match.group(1).upper()
            box = re.search(r"box=\[\[?(\d+),\s*(\d+),\s*(\d+),\s*(\d+)\]?\]", step)
            if box:
                x1, y1, x2, y2 = [int(v) for v in box.groups()]
                cx = int((x1 + x2) / 2 * img_w / 1000)
                cy = int((y1 + y2) / 2 * img_h / 1000)
            else:
                cx, cy = None, None

            result = {"description": action_desc or step}

            if operation == "CLICK":
                result["action"] = "click"
                if cx is not None:
                    result["x"], result["y"] = cx, cy
            elif operation == "TYPE":
                result["action"] = "type"
                tm = re.search(r"text=['\"]([^'\"]*)['\"]", step)
                result["text"] = tm.group(1) if tm else ""
                if cx is not None:
                    result["x"], result["y"] = cx, cy
            elif operation in ("SCROLL_DOWN", "SCROLL_UP"):
                result["action"] = "scroll"
                result["direction"] = "down" if "DOWN" in operation else "up"
                result["clicks"] = 3
                if cx is not None:
                    result["x"], result["y"] = cx, cy
            elif operation == "PRESS":
                result["action"] = "key"
                km = re.search(r"key=['\"]([^'\"]*)['\"]", step)
                result["key"] = km.group(1) if km else "Enter"
            elif operation == "HOVER":
                result["action"] = "click"
                if cx is not None:
                    result["x"], result["y"] = cx, cy
            elif operation == "WAIT":
                result["action"] = "done"
                result["description"] = "Waiting..."
            else:
                result["action"] = operation.lower()
                if cx is not None:
                    result["x"], result["y"] = cx, cy
            return result

    # ── Fallback: non-structured format parsing ──

    # Click [[x,y]]
    m = re.match(
        r'(?i)(click|double[_ ]?click|right[_ ]?click)\s*'
        r'\[\[?\s*(\d+)\s*[,\s]\s*(\d+)\s*\]?\]?\s*(.*)', text)
    if m:
        rx, ry = int(m.group(2)), int(m.group(3))
        desc = m.group(4).strip() or m.group(1)
        x = int(rx / 1000 * img_w) if rx <= 1000 else rx
        y = int(ry / 1000 * img_h) if ry <= 1000 else ry
        return {"action": "click", "x": x, "y": y, "description": desc}

    # Type [[x,y]] text
    m = re.match(r'(?i)type\s*\[\[?\s*(\d+)\s*[,\s]\s*(\d+)\s*\]?\]?\s+(.*)', text)
    if m:
        rx, ry = int(m.group(1)), int(m.group(2))
        x = int(rx / 1000 * img_w) if rx <= 1000 else rx
        y = int(ry / 1000 * img_h) if ry <= 1000 else ry
        return {"action": "type", "x": x, "y": y,
                "text": m.group(3).strip(),
                "description": f"typing: {m.group(3).strip()[:50]}"}

    m = re.match(r'(?i)type\s+(.*)', text)
    if m:
        return {"action": "type", "text": m.group(1).strip(),
                "description": f"typing: {m.group(1).strip()[:50]}"}

    m = re.match(r'(?i)(?:press|key)\s+(.*)', text)
    if m:
        return {"action": "key", "key": m.group(1).strip(),
                "description": f"pressing {m.group(1).strip()}"}

    m = re.match(r'(?i)scroll\s+(up|down|left|right)\s*(\d*)', text)
    if m:
        return {"action": "scroll", "direction": m.group(1).lower(),
                "clicks": int(m.group(2)) if m.group(2) else 3,
                "description": f"scrolling {m.group(1).lower()}"}

    if re.match(r'(?i)(done|status:\s*done|task\s*complete)', text):
        return {"action": "done", "description": "task complete"}

    return None


def run_vision_inference(image_b64: str, goal: str, step_history: list,
                         screen_w: int, screen_h: int,
                         generation_config=None) -> dict:
    """Screenshot + goal -> action JSON via CogAgent.

    Uses official tokenizer.apply_chat_template() for input construction
    and matches the prompt format from CogAgent's client.py / cli_demo.py.
    """
    # Build CogAgent task prompt (matches official format)
    history_str = "\nHistory steps: "
    for i, s in enumerate(step_history):
        action = s.get('action', '?')
        desc = s.get('description', '')
        ok = 'ok' if s.get('success') else 'fail'
        history_str += f"\n{i}. [{action}] {desc} {ok}"

    query = (
        f"Task: {goal}{history_str}\n"
        f"(Platform: WIN)\n"
        f"(Answer in Action-Operation-Sensitive format.)\n"
    )

    # Decode screenshot
    img_bytes = base64.b64decode(image_b64)
    image = Image.open(io.BytesIO(img_bytes)).convert("RGB")

    print(f"  Building inputs for: {goal[:60]}...")

    # Official API: tokenizer.apply_chat_template
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "image": image, "content": query}],
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
    ).to(INPUT_DEVICE)

    print(f"   input_ids: {inputs['input_ids'].shape}")

    # Generate (use GenerationConfig — required for transformers>=4.49)
    with torch.no_grad():
        try:
            _gc = generation_config if generation_config is not None else GEN_CONFIG
            outputs = model.generate(**inputs, generation_config=_gc, **GEN_EXTRA)
        except RuntimeError as e:
            if 'CUDA' in str(e) or 'assert' in str(e):
                print(f"  ❌ CUDA error: {e}")
                try:
                    torch.cuda.empty_cache()
                except Exception:
                    pass
                return {"action": "fail",
                        "description": f"CUDA error: {str(e)[:100]}"}
            raise

    input_len = inputs["input_ids"].shape[-1]
    new_tokens = outputs[0][input_len:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    print(f"  🤖 Output: {response[:200]}")

    try:
        torch.cuda.empty_cache()
    except Exception:
        pass

    # Parse response
    result = _parse_cogagent_native(response, screen_w, screen_h)
    if result:
        print(f"  ✅ Parsed: {result['action']}")
        return result

    # JSON fallback
    try:
        jm = re.search(r'\{[^{}]*\}', response)
        if jm:
            result = json.loads(jm.group())
            if 'action' in result:
                print(f"  ✅ JSON: {result['action']}")
                return result
    except (json.JSONDecodeError, AttributeError):
        pass

    _lower = response.lower()
    if any(kw in _lower for kw in ["done", "finish", "complet", "success"]):
        return {"action": "done", "description": response[:200]}

    print(f"  ⚠️ Unparseable response")
    return {"action": "fail", "description": f"Unparseable: {response[:200]}"}


print("✅ Vision inference ready")
print(f"   Uses official tokenizer.apply_chat_template() API")
print(f"   Safety: InvalidScoreLogitsProcessor + top_k=1 (greedy)")

## 3.5 Validation Tests
Run these tests to confirm CogAgent produces **real structured output** (not garbage).
- Test A: Text-only coherence
- Test B: Image + CogAgent task prompt → structured Action/Operation output
- Test C: Full pipeline via `run_vision_inference()`

All tests must pass before starting the server.


In [ ]:
# ============================================================
# Cell 3.5: Validation Tests — confirm model output quality
# ============================================================
import base64, io, time
from PIL import Image as PILImage

_all_pass = True

# ═══════════════════════════════════════════════════════════════
#  TEST A: Text-only — model should produce coherent English
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print("TEST A: Text-only coherence check")
print("=" * 60)
try:
    _a_inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": "What is 2 + 3? Answer with just the number."}],
        add_generation_prompt=True, tokenize=True,
        return_tensors="pt", return_dict=True,
    ).to(INPUT_DEVICE)

    with torch.no_grad():
        _a_out = model.generate(**_a_inputs, generation_config=WARMUP_GEN_CONFIG, **GEN_EXTRA)
    _a_ids = _a_out[0][_a_inputs["input_ids"].shape[-1]:].tolist()
    _a_text = tokenizer.decode(_a_ids, skip_special_tokens=True).strip()

    print(f"  Output IDs : {_a_ids[:15]}")
    print(f"  Output text: '{_a_text}'")

    _a_has_alnum = any(c.isalnum() for c in _a_text)
    _a_not_garbage = _a_text.count('!') < len(_a_text) * 0.5 if _a_text else False

    if _a_has_alnum and _a_not_garbage:
        print("  ✅ TEST A PASSED — coherent text output")
    else:
        print("  ❌ TEST A FAILED — output looks like garbage")
        _all_pass = False
except Exception as e:
    print(f"  ❌ TEST A ERROR: {e}")
    import traceback; traceback.print_exc()
    _all_pass = False

# ═══════════════════════════════════════════════════════════════
#  TEST B: Image + CogAgent task prompt → structured output
# ═══════════════════════════════════════════════════════════════
print()
gc.collect(); torch.cuda.empty_cache()
print("=" * 60)
print("TEST B: Image + CogAgent task prompt (structured output)")
print("=" * 60)
try:
    # Synthetic desktop: white bg + blue taskbar + blue button
    _b_img = PILImage.new("RGB", (1920, 1080), (240, 240, 240))
    from PIL import ImageDraw
    _draw = ImageDraw.Draw(_b_img)
    _draw.rectangle([0, 1040, 1920, 1080], fill=(0, 50, 120))       # taskbar
    _draw.rectangle([800, 400, 1120, 460], fill=(0, 120, 215))      # blue button
    _draw.rectangle([850, 420, 1070, 440], fill=(255, 255, 255))    # button text area

    _b_query = (
        "Task: Click the blue button in the center of the screen.\n"
        "History steps: \n"
        "(Platform: WIN)\n"
        "(Answer in Action-Operation-Sensitive format.)\n"
    )

    _b_inputs = tokenizer.apply_chat_template(
        [{"role": "user", "image": _b_img, "content": _b_query}],
        add_generation_prompt=True, tokenize=True,
        return_tensors="pt", return_dict=True,
    ).to(INPUT_DEVICE)

    print(f"  Input shape: {_b_inputs['input_ids'].shape}, keys: {list(_b_inputs.keys())}")

    _b_start = time.time()
    with torch.no_grad():
        _b_out = model.generate(**_b_inputs, generation_config=TEST_GEN_CONFIG, **GEN_EXTRA)
    _b_elapsed = time.time() - _b_start

    _b_out_ids = _b_out[0][_b_inputs['input_ids'].shape[-1]:].tolist()
    _b_text = tokenizer.decode(_b_out_ids, skip_special_tokens=True).strip()

    print(f"  Output IDs (first 30): {_b_out_ids[:30]}")
    print(f"  Output text:")
    for _line in _b_text.split('\n'):
        print(f"    | {_line}")
    print(f"  Tokens: {len(_b_out_ids)} in {_b_elapsed:.1f}s ({len(_b_out_ids)/_b_elapsed:.1f} tok/s)")

    _b_lower = _b_text.lower()
    _b_has_status = "status:" in _b_lower
    _b_has_action = "action:" in _b_lower
    _b_has_grounded = "grounded operation:" in _b_lower or "operation:" in _b_lower
    _b_has_alnum = sum(c.isalnum() for c in _b_text) > 10
    _b_not_garbage = _b_text.count('!') < len(_b_text) * 0.3 if _b_text else False

    checks = []
    if _b_has_status: checks.append("Status")
    if _b_has_action: checks.append("Action")
    if _b_has_grounded: checks.append("Grounded Operation")

    if _b_has_alnum and _b_not_garbage and len(checks) >= 2:
        print(f"  ✅ TEST B PASSED — found: {', '.join(checks)}")
    elif _b_has_alnum and _b_not_garbage:
        print(f"  ⚠️ TEST B PARTIAL — coherent but missing structured format")
        print(f"     Found: {checks if checks else 'none of Status/Action/Grounded Op'}")
        print(f"     Synthetic images are harder than real screenshots — may still work")
    else:
        print(f"  ❌ TEST B FAILED — garbage output")
        _all_pass = False

    del _b_img
except Exception as e:
    print(f"  ❌ TEST B ERROR: {e}")
    import traceback; traceback.print_exc()
    _all_pass = False

# ═══════════════════════════════════════════════════════════════
#  TEST C: Full pipeline through run_vision_inference()
# ═══════════════════════════════════════════════════════════════
print()
gc.collect(); torch.cuda.empty_cache()
print("=" * 60)
print("TEST C: Full pipeline via run_vision_inference()")
print("=" * 60)
try:
    _c_img = PILImage.new("RGB", (1920, 1080), (255, 255, 255))
    _c_draw = ImageDraw.Draw(_c_img)
    _c_draw.rectangle([400, 200, 1520, 240], fill=(230, 230, 230), outline=(180, 180, 180))

    _c_buf = io.BytesIO()
    _c_img.save(_c_buf, format="PNG")
    _c_b64 = base64.b64encode(_c_buf.getvalue()).decode()

    _c_start = time.time()
    _c_result = run_vision_inference(
        image_b64=_c_b64,
        goal="Click on the search bar",
        step_history=[],
        screen_w=1920,
        screen_h=1080,
        generation_config=TEST_GEN_CONFIG,
    )
    _c_elapsed = time.time() - _c_start

    print(f"  Result: {_c_result}")
    print(f"  Time:   {_c_elapsed:.1f}s")

    _c_action = _c_result.get("action", "")
    if _c_action in ("click", "type", "scroll", "key", "done"):
        print(f"  ✅ TEST C PASSED — got valid action: '{_c_action}'")
        if "x" in _c_result and "y" in _c_result:
            print(f"     Coordinates: ({_c_result['x']}, {_c_result['y']})")
        if "description" in _c_result:
            print(f"     Description: {_c_result['description'][:100]}")
    elif _c_action == "fail":
        desc = _c_result.get("description", "")
        if "Unparseable" in desc:
            print(f"  ⚠️ TEST C PARTIAL — model responded but format not recognized")
            print(f"     Raw: {desc[:200]}")
        else:
            print(f"  ❌ TEST C FAILED — {desc[:200]}")
            _all_pass = False
    else:
        print(f"  ❌ TEST C FAILED — unexpected action: '{_c_action}'")
        _all_pass = False

    del _c_img
except Exception as e:
    print(f"  ❌ TEST C ERROR: {e}")
    import traceback; traceback.print_exc()
    _all_pass = False

# ═══════════════════════════════════════════════════════════════
#  Summary
# ═══════════════════════════════════════════════════════════════
print()
print("=" * 60)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    for _gi in range(torch.cuda.device_count()):
        _used = torch.cuda.memory_allocated(_gi) / 1e9
        _total = torch.cuda.get_device_properties(_gi).total_memory / 1e9
        print(f"📊 GPU {_gi} VRAM: {_used:.1f} / {_total:.1f} GB")

if _all_pass:
    print("🎉 ALL TESTS PASSED — model is producing quality output!")
    print("   Safe to proceed to Cell 4 (FastAPI server).")
else:
    print("⚠️ SOME TESTS FAILED — check output above.")
    print("   The model may still work for real screenshots —")
    print("   synthetic test images are harder than real ones.")
    print("   Proceed to Cell 4 if Test A passed (coherent text).")
print("=" * 60)

## 5. Start FastAPI Server with ngrok


In [ ]:
# ============================================================
# Cell 4: FastAPI Server + ngrok Tunnel
# ============================================================
import nest_asyncio
nest_asyncio.apply()

import asyncio
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Dict, Optional
import uvicorn
import time

# ── FastAPI App ──
app = FastAPI(title="Pecifics CogAgent Vision Server")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

class VisionActRequest(BaseModel):
    screenshot: str
    goal: str
    step_history: List[Dict] = []
    screen_width: Optional[int] = 1920
    screen_height: Optional[int] = 1080

@app.get("/")
async def root():
    return {"service": "Pecifics CogAgent Vision Server", "endpoints": ["/health", "/vision_act"]}

@app.get("/health")
async def health():
    return {
        "status": "ok",
        "model": MODEL_ID,
        "dtype": str(COMPUTE_DTYPE),
        "service": "cogagent-vision",
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "vram_used_gb": round(torch.cuda.memory_allocated() / 1e9, 1) if torch.cuda.is_available() else 0,
    }

@app.post("/vision_act")
async def vision_act(req: VisionActRequest):
    """Analyze screenshot, return next mouse/keyboard action with coordinates."""
    start = time.time()
    try:
        result = run_vision_inference(
            image_b64=req.screenshot,
            goal=req.goal,
            step_history=req.step_history,
            screen_w=req.screen_width,
            screen_h=req.screen_height,
        )
        elapsed = time.time() - start
        print(f"🎯 vision_act ({elapsed:.1f}s) → {result.get('action')}: {result.get('description', '')[:80]}")
        return result
    except Exception as e:
        elapsed = time.time() - start
        print(f"❌ vision_act error ({elapsed:.1f}s): {e}")
        import traceback
        traceback.print_exc()
        return {"action": "fail", "description": f"Vision error: {str(e)}"}

# ── ngrok Tunnel ──
VISION_PORT = 8002

# ══════════════════════════════════════════════════════════════
# OPTION A: Paste your ngrok token directly here (easiest fix)
# Get it from: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN_MANUAL = ""  # ← Paste your token inside the quotes
# ══════════════════════════════════════════════════════════════

NGROK_TOKEN = NGROK_TOKEN_MANUAL

# OPTION B: Try reading from Kaggle Secrets (requires toggling access ON)
if not NGROK_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        NGROK_TOKEN = secrets.get_secret("NGROK_TOKEN")
        print(f"✅ Got NGROK_TOKEN from Kaggle Secrets (length: {len(NGROK_TOKEN)})")
    except Exception as e:
        print(f"⚠️ Kaggle Secrets failed: {type(e).__name__}: {e}")
        print("   → Make sure the secret is named exactly 'NGROK_TOKEN'")
        print("   → Make sure you toggled the checkbox to grant this notebook access")
        import os
        NGROK_TOKEN = os.environ.get("NGROK_TOKEN", "")
        if NGROK_TOKEN:
            print(f"✅ Got NGROK_TOKEN from environment variable")

if not NGROK_TOKEN:
    print("=" * 60)
    print("❌ ngrok auth token not found! Try one of these:")
    print()
    print("   FIX 1 (easiest): Paste your token directly in this cell:")
    print('          NGROK_TOKEN_MANUAL = "your_token_here"')
    print()
    print("   FIX 2: Add Kaggle Secret named NGROK_TOKEN:")
    print("          Add-ons (left panel) → Secrets → Add")
    print("          ⚠️ Toggle the checkbox to grant notebook access!")
    print()
    print("   Get your token: https://dashboard.ngrok.com/get-started/your-authtoken")
    print("=" * 60)
    raise SystemExit("Missing NGROK_TOKEN — see instructions above")

try:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_TOKEN)
    public_url = ngrok.connect(VISION_PORT)

    print()
    print("=" * 60)
    print("🚀 CogAgent Vision Server is LIVE!")
    print(f"📡 Public URL: {public_url}")
    print(f"🏠 Local URL:  http://localhost:{VISION_PORT}")
    print()
    print("👉 Set this in your .env file:")
    print(f"   COGAGENT_URL={public_url}")
    print()
    print("👉 Or paste this URL in Pecifics Settings → CogAgent URL")
    print("=" * 60)

except Exception as e:
    if "Missing NGROK_TOKEN" in str(e):
        raise
    print(f"❌ ngrok tunnel failed: {e}")
    print(f"   Server will still run on http://localhost:{VISION_PORT}")
    print(f"   Use SSH port-forwarding or localtunnel as alternative.")

# Run server (nest_asyncio makes this work in Jupyter)
config = uvicorn.Config(app, host="0.0.0.0", port=VISION_PORT, log_level="info")
server = uvicorn.Server(config)
asyncio.get_event_loop().run_until_complete(server.serve())